In [3]:
import pandas as pd
import numpy as np

In [4]:
# ── 1. LOAD ───────────────────────────────────────────────────────────────────
print("=" * 60)
print("STEP 1: Loading raw data")
print("=" * 60)

df     = pd.read_excel("yamato_holdings_fpa.xlsx", sheet_name="Raw_Division_Data")
interco= pd.read_excel("yamato_holdings_fpa.xlsx", sheet_name="Interco_Elimination")
fx     = pd.read_excel("yamato_holdings_fpa.xlsx", sheet_name="FX_Reference")
issues = pd.read_excel("yamato_holdings_fpa.xlsx", sheet_name="Data_Quality_Log")

print(f"Loaded {len(df)} rows x {len(df.columns)} columns")
print(f"Divisions : {df['division'].unique().tolist()}")
print(f"Date range: {df['ds'].min()} → {df['ds'].max()}")
print()

STEP 1: Loading raw data
Loaded 72 rows x 20 columns
Divisions : ['Japan Domestic', 'Asia Export', 'Corporate HQ']
Date range: 2022-04-01 → 2024-03-01



In [5]:
# ── 2. DATA QUALITY CHECK ─────────────────────────────────────────────────────
print("=" * 60)
print("STEP 2: Data quality check")
print("=" * 60)

missing = df.isnull().sum()
missing = missing[missing > 0]
print("Missing values found:")
print(missing.to_string())
print()

flagged = df[df["data_issue"].notna()][["ds","division","data_issue"]]
print(f"Flagged rows: {len(flagged)}")
print(flagged.to_string(index=False))
print()

STEP 2: Data quality check
Missing values found:
usd_jpy_rate    48
exp_it_jpy       1
data_issue      71

Flagged rows: 1
        ds    division                                    data_issue
2022-06-01 Asia Export missing IT expense — not reported by division



In [6]:
# ── 3. CLEAN ─────────────────────────────────────────────────────────────────
print("=" * 60)
print("STEP 3: Cleaning data")
print("=" * 60)

df["ds"] = pd.to_datetime(df["ds"])

# Fix missing IT expense via linear interpolation within division
missing_mask = df["exp_it_jpy"].isnull()
df["exp_it_jpy"] = df.groupby("division")["exp_it_jpy"].transform(
    lambda x: x.interpolate(method="linear")
)
print(f"Interpolated {missing_mask.sum()} missing IT expense value(s)")

# HQ is a cost centre — COGS is structurally zero, not missing
df["exp_cogs_jpy"] = df["exp_cogs_jpy"].fillna(0)
print("Filled COGS with 0 for Corporate HQ (cost centre — expected, not a data error)")
print()

STEP 3: Cleaning data
Interpolated 1 missing IT expense value(s)
Filled COGS with 0 for Corporate HQ (cost centre — expected, not a data error)



In [7]:
# ── 4. INTERCOMPANY ELIMINATION ───────────────────────────────────────────────
print("=" * 60)
print("STEP 4: Intercompany elimination")
print("=" * 60)

# interco_charge_jpy is positive for payer divisions, negative for HQ (receiver)
# Net per month should be zero — any balance = reporting discrepancy
interco_check = df.groupby("ds")["interco_charge_jpy"].sum()
discrepancies = interco_check[abs(interco_check) > 1_000]

if len(discrepancies) == 0:
    print("✓ All intercompany charges net to zero across divisions")
else:
    print(f"⚠  {len(discrepancies)} month(s) with interco discrepancies — investigate:")
    print(discrepancies.to_string())

# External revenue = strip out interco income/charges before consolidating
df["rev_external_jpy"] = df["total_rev_jpy"] - df["interco_charge_jpy"].clip(lower=0)
print()

STEP 4: Intercompany elimination
✓ All intercompany charges net to zero across divisions



In [8]:


# ── 5. CONSOLIDATE ───────────────────────────────────────────────────────────
print("=" * 60)
print("STEP 5: Consolidating across all divisions")
print("=" * 60)

consolidated = df.groupby("ds").agg(
    total_rev_jpy      =("rev_external_jpy",  "sum"),
    total_exp_jpy      =("total_exp_jpy",      "sum"),
    exp_headcount_jpy  =("exp_headcount_jpy",  "sum"),
    exp_it_jpy         =("exp_it_jpy",         "sum"),
    exp_cogs_jpy       =("exp_cogs_jpy",       "sum"),
    exp_compliance_jpy =("exp_compliance_jpy", "sum"),
    budget_rev_jpy     =("budget_rev_jpy",     "sum"),
    budget_exp_jpy     =("budget_exp_jpy",     "sum"),
).reset_index()

consolidated["ebitda_jpy"]        = consolidated["total_rev_jpy"] - consolidated["total_exp_jpy"]
consolidated["budget_ebitda_jpy"] = consolidated["budget_rev_jpy"] - consolidated["budget_exp_jpy"]

# Variance: use absolute budget as denominator to avoid division-by-near-zero explosion
consolidated["variance_vs_budget_jpy"] = consolidated["ebitda_jpy"] - consolidated["budget_ebitda_jpy"]
consolidated["ebitda_vs_budget_pct"]   = (
    consolidated["variance_vs_budget_jpy"] / consolidated["budget_rev_jpy"] * 100
).round(2)  # normalised to budget revenue — stable denominator

# Prophet columns
consolidated["y"]  = consolidated["ebitda_jpy"]

print(f"Consolidated to {len(consolidated)} monthly rows")
print()
print(consolidated[["ds","total_rev_jpy","total_exp_jpy","ebitda_jpy","ebitda_vs_budget_pct"]].to_string(index=False))
print()

STEP 5: Consolidating across all divisions
Consolidated to 24 monthly rows

        ds  total_rev_jpy  total_exp_jpy  ebitda_jpy  ebitda_vs_budget_pct
2022-04-01     2013139038     2091264246   -78125208                 -8.55
2022-05-01     2015834499     2091865832   -76031333                 -7.81
2022-06-01     2091857265     2128370210   -36512945                 -4.97
2022-07-01     1855521851     2062528056  -207006205                 -9.36
2022-08-01     1816044040     2090809559  -274765519                 -8.74
2022-09-01     2085634371     2155201853   -69567482                -11.31
2022-10-01     2131040867     2275211302  -144170435                 -4.64
2022-11-01     2157086621     2218660559   -61573938                 -7.36
2022-12-01     2077939250     2183897866  -105958616                 -8.30
2023-01-01     2438841016     2393316846    45524170                 -6.21
2023-02-01     2361399861     2275108941    86290920                -10.20
2023-03-01     247224971

In [9]:
# ── 6. ANOMALY FLAG ───────────────────────────────────────────────────────────
print("=" * 60)
print("STEP 6: Flagging budget variance anomalies")
print("=" * 60)

# Flag months where EBITDA deviates more than 5% of budget revenue
THRESHOLD = 5.0
anomalies = consolidated[abs(consolidated["ebitda_vs_budget_pct"]) > THRESHOLD].copy()

print(f"Threshold: EBITDA variance > {THRESHOLD}% of consolidated budget revenue")
print(f"Anomalies detected: {len(anomalies)}")
print()
print(anomalies[["ds","ebitda_jpy","budget_ebitda_jpy","ebitda_vs_budget_pct"]].to_string(index=False))
print()

# Save for Phase 3
consolidated.to_csv("yamato_consolidated.csv", index=False)
print("✓ Saved yamato_consolidated.csv — ready for Phase 3 (forecasting)")


STEP 6: Flagging budget variance anomalies
Threshold: EBITDA variance > 5.0% of consolidated budget revenue
Anomalies detected: 19

        ds  ebitda_jpy  budget_ebitda_jpy  ebitda_vs_budget_pct
2022-04-01   -78125208          107952409                 -8.55
2022-05-01   -76031333           86962012                 -7.81
2022-07-01  -207006205          -20032009                 -9.36
2022-08-01  -274765519         -103450898                 -8.74
2022-09-01   -69567482          184816077                -11.31
2022-11-01   -61573938          108538465                 -7.36
2022-12-01  -105958616           72690406                 -8.30
2023-01-01    45524170          209265801                 -6.21
2023-02-01    86290920          353093901                -10.20
2023-03-01   160313306          446417078                -10.72
2023-04-01   -50693449          106103132                 -7.18
2023-05-01   -79834712           73847184                 -6.84
2023-07-01  -152081210          -115

In [10]:
from prophet import Prophet
print("Prophet ready")

Prophet ready
